In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix



In [3]:
df = sns.load_dataset('titanic')
print(df.head())

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


In [4]:
data = df.drop(['deck','alive','who','adult_male','class','embark_town'],axis=1)

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data['survived'].value_counts(normalize=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

to_categorical_columns = ['sibsp','pclass','parch']
data[to_categorical_columns] = data[to_categorical_columns].astype('category')

numeric_columns = data.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

# pandas 최신 버전 대응
for col in data.select_dtypes(include=['object', 'string']).columns:
    data[col] = data[col].astype('category')

categorical_columns = data.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

numeric_columns.remove('survived')

print("Numeric Columns 단변량 분석\n")
for col in numeric_columns:
    plt.figure(figsize=(12, 4))

    # Histogram
    plt.subplot(1, 2, 1)
    sns.histplot(
        data[col],
        kde=True
    )
    plt.title(f'{col} Histogram')

    # Boxplot
    plt.subplot(1, 2, 2)
    sns.boxplot(
        x=data[col]
    )
    plt.title(f'{col} Boxplot')

    plt.tight_layout()
    plt.show()

print("="*50)
print("categorical columns 단변량 분석\n")
for col in categorical_columns:
    plt.figure(figsize=(8, 4))

    sns.countplot(
        x=data[col],
        order=data[col].value_counts().index
    )

    plt.title(f'{col} Countplot')
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

흠 일단 자료들의 분포를 보니 fare hist는 오른쪽으로 꼬리가 아주 긴 분포네, log변환 해보고싶긴하다.
 그리고 categori들 pclass, sibsp, parch는 각 자료간에 위계가 존재하기때문에 ordinal encoding 고려해볼만하고,
 그외는 onehotencoding 해주면 될듯

# Titanic Dataset 컬럼 설명 (현재 사용 기준)

## Target
- **survived**
  - 생존 여부
  - `0 = 사망`, `1 = 생존`

---

## Numeric Features

- **age**
  - 승객 나이
  - 일부 결측치 존재
  - 연속형 변수

- **fare**
  - 티켓 요금
  - 오른쪽 꼬리가 긴 분포 (log 변환 고려 가능)

- **sibsp**
  - 함께 탑승한 형제자매 / 배우자 수
  - 가족 규모 관련 feature

- **parch**
  - 함께 탑승한 부모 / 자녀 수
  - 가족 규모 관련 feature

---

## Categorical Features

- **pclass**
  - 객실 등급
  - `1 = 1등석`, `2 = 2등석`, `3 = 3등석`
  - 사회경제적 수준 반영 가능
  - 순서형 특성 존재

- **sex**
  - 성별
  - `male / female`

- **embarked**
  - 탑승 항구
  - `C = Cherbourg`
  - `Q = Queenstown`
  - `S = Southampton`

---

## 제거한 컬럼들

- **deck**
  - 객실 층 정보
  - 결측치 과다

- **alive**
  - survived와 중복

- **class**
  - pclass와 중복

- **who**
  - sex + age 기반 파생 변수

- **adult_male**
  - sex + age 중복성 높음

- **embark_town**
  - embarked와 중복

---

## Feature Engineering 후보

- **family_size**
```python
sibsp + parch + 1

In [ ]:
data = df.drop(['deck','alive','who','adult_male','class','embark_town'],axis=1)
data['alone'] = data['alone'].astype(int)

In [ ]:
#파생변수 설정
# data['family_size'] = data['sibsp'] + data['parch'] + 1

In [ ]:
X = data.drop('survived', axis=1)
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
# feature selection
numeric_features = ['age', 'fare', 'sibsp', 'parch','alone']
categorical_features = ['pclass', 'sex', 'embarked']

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

baseline_pipe = Pipeline([
    ('preprocessor', preprocessor),
    (
        'model', LogisticRegression(max_iter=1000,
                                    random_state=42)
    )
])

cv_scores = cross_validate(
    baseline_pipe,
    X_train,
    y_train,
    cv=5,
    scoring=['accuracy','f1','precision','recall','roc_auc']
)
print("Accuracy : ",cv_scores['test_accuracy'].mean())
print("F1 Score : ",cv_scores['test_f1'].mean())
print("Precision : ",cv_scores['test_precision'].mean())
print("Recall : ",cv_scores['test_recall'].mean())
print("ROC AUC : ",cv_scores['test_roc_auc'].mean())



In [ ]:
baseline_pipe.fit(X_train,y_train)
y_pred = baseline_pipe.predict(X_test)

print("test_Acc:", baseline_pipe.score(X_test,y_test))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
def evaluate_pipe (pipe, X_train, y_train, X_test, y_test):
    cv_scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring=['accuracy','f1','precision','recall','roc_auc']
    )
    print("Accuracy : ",cv_scores['test_accuracy'].mean())
    print("F1 Score : ",cv_scores['test_f1'].mean())
    print("Precision : ",cv_scores['test_precision'].mean())
    print("Recall : ",cv_scores['test_recall'].mean())
    print("ROC AUC : ",cv_scores['test_roc_auc'].mean())

    pipe.fit(X_train,y_train)
    y_pred = pipe.predict(X_test)
    print("test_Acc:", pipe.score(X_test,y_test))
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

In [ ]:
# fare feature log변환 해보자
numeric_features = ['age', 'sibsp', 'parch','alone']
fare_feature = ['fare']
categorical_features = ['pclass', 'sex', 'embarked']

In [ ]:
fare_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, validate=False)),
    ('scaler', StandardScaler())
])

log_preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
    ('fare', fare_transformer, fare_feature)
])

log_pipe = Pipeline([
    ('preprocessor', log_preprocessor),
    ('model', LogisticRegression(max_iter=1000,))
])

evaluate_pipe(log_pipe, X_train, y_train, X_test, y_test)

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from my_ml_kit import (
    run_model_candidates,
    compare_results,
    print_test_report
)

In [5]:
X = data.drop('survived', axis=1)
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [11]:
numeric_features = ['age', 'fare', 'sibsp', 'parch', 'alone']
categorical_features = ['pclass', 'sex', 'embarked']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [23]:
basic_models = {
    'logistic': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    'decision_tree': DecisionTreeClassifier(
        random_state=42
    ),

    'knn': KNeighborsClassifier(),

    'gaussian_nb': GaussianNB()
}

In [24]:
results = []

results, trained_pipes = run_model_candidates(
    models=basic_models,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    results=results,
    cv=5
)

Running: logistic
Running: decision_tree
Running: knn
Running: gaussian_nb


In [28]:
results_df = compare_results(
    results,
    sort_by='cv_accuracy',
)

results_df

,name,cv_accuracy,cv_f1,cv_precision,cv_recall,cv_roc_auc,test_accuracy,test_f1,test_precision,test_recall,test_roc_auc
0,logistic,0.793627,0.720169,0.747602,0.699865,0.855856,0.815642,0.740157,0.810345,0.681159,0.850461
1,knn,0.806264,0.736682,0.764519,0.714411,0.851255,0.815642,0.744186,0.800000,0.695652,0.850329
2,gaussian_nb,0.785167,0.728319,0.708102,0.751044,0.836095,0.765363,0.695652,0.695652,0.695652,0.830567
3,decision_tree,0.788023,0.722218,0.725209,0.722020,0.781000,0.821229,0.757576,0.793651,0.724638,0.798353


In [16]:
for model_name in basic_models.keys():
    print(f"\n===== {model_name} =====")

    print_test_report(
        trained_pipes[model_name],
        X_test,
        y_test
    )


===== logistic =====
[[99 11]
 [22 47]]
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.68      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179


===== decision_tree =====
[[97 13]
 [19 50]]
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       110
           1       0.79      0.72      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179


===== knn =====
[[98 12]
 [21 48]]
              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   ma

In [ ]:
baseline_pipe.get_params()

In [ ]:
trained_pipes['logistic'].get_params()

In [ ]:
#autogluon